# Practice: What Company Does 'Government' Keep?

**Computational Text Analysis Pack · Module 05 — Concordance and Collocation**

*Technique:* Collocation to concordance and back; a reading grounded in counts

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/03-comparing-corpora/05e_practice_collocation.ipynb)

Data: `masakhanews.csv`, `colonial_texts.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv', 'colonial_texts.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
df_colonial = pd.read_csv('colonial_texts.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## The question

> **Is 'government' written about differently in contemporary African news than
> in pre-1929 European writing about Africa — and can a concordance show it?**

This is the module's core move: a count points at passages, and the passages are
what you actually read. The count is the index, not the argument.

### How to use this notebook

This is a worked case, not a tutorial. Run it once to see the shape of the
argument, then change the question at the top to one of your own and run it
again. The cells are written so that swapping the question is the only edit you
need.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
import math
NODE = "government"      # <- change this
WINDOW = 5

def collocates(docs, node=NODE, window=WINDOW, min_count=8):
    uni, pair = Counter(), Counter()
    for d in docs:
        toks = tokenise(d)
        uni.update(toks)
        for i, w in enumerate(toks):
            if w != node: continue
            for j in range(max(0, i-window), min(len(toks), i+window+1)):
                if j != i: pair[toks[j]] += 1
    N = sum(uni.values())
    out = []
    for w, joint in pair.items():
        if joint < min_count or uni[w] < 20: continue
        dice = 2*joint/(uni[node]+uni[w])
        out.append({"word": w, "joint": joint, "log_dice": 14+math.log2(dice)})
    return pd.DataFrame(out).sort_values("log_dice", ascending=False), uni

modern_docs = [str(t) for t in df[df["language"] == "English"]["text"]]
old_docs = [str(t) for t in df_colonial["text"]]
mod, umod = collocates(modern_docs)
old, uold = collocates(old_docs)
print(f"'{NODE}': {umod[NODE]:,} occurrences in modern news, "
      f"{uold[NODE]:,} in pre-1929 prose\n")
print("MODERN top collocates :", ", ".join(mod.head(12)["word"]))
print("PRE-1929 top collocates:", ", ".join(old.head(12)["word"]))

In [ ]:
shared = set(mod.head(25)["word"]) & set(old.head(25)["word"])
only_new = [w for w in mod.head(25)["word"] if w not in shared]
only_old = [w for w in old.head(25)["word"] if w not in shared]
print(f"shared by both periods : {sorted(shared)}")
print(f"\nmodern only            : {only_new[:12]}")
print(f"pre-1929 only          : {only_old[:12]}")
print()
print("Resist reading those lists as an argument. They are a list of places to")
print("look, and the looking is next.")

In [ ]:
# Now the concordance. This is where the claim gets made or abandoned.
def kwic(docs, node, partner=None, width=42, limit=6):
    out = []
    for d in docs:
        toks = tokenise(d)
        for i, w in enumerate(toks):
            if w != node: continue
            lo, hi = max(0, i-12), min(len(toks), i+13)
            window = toks[lo:hi]
            if partner and partner not in window: continue
            left = " ".join(toks[lo:i])[-width:]
            right = " ".join(toks[i+1:hi])[:width]
            out.append(f"{left:>{width}}  {node.upper():^12}  {right:<{width}}")
            if len(out) >= limit: return out
    return out

probe = only_old[0] if only_old else (only_new[0] if only_new else None)
if probe:
    print(f"PRE-1929, '{NODE}' near '{probe}':\n")
    for line in kwic(old_docs, NODE, probe): print(line)
    print()
probe2 = only_new[0] if only_new else None
if probe2:
    print(f"MODERN, '{NODE}' near '{probe2}':\n")
    for line in kwic(modern_docs, NODE, probe2): print(line)

## What this licenses

Read the concordance lines above before writing anything. Three questions:

1. **Is the collocate doing what the list implied?** Often it is incidental —
   a word that co-occurs for grammatical rather than semantic reasons.
2. **Do the lines cluster into senses?** If *government* near a given word means
   two different things across the lines, one collocate is hiding two patterns.
3. **How many distinct documents?** Six lines from two articles is not evidence
   about a period.

**Defensible:** "In this corpus, *government* co-occurs with X in pre-1929
writing and with Y in contemporary news; the concordance lines show the former
used in [a specific construction]."

**Not defensible:** a claim about how colonial discourse *thought* about
government. The corpus is a Gutenberg keyword search, and module 22c is where
that limit gets stated properly.

### 🔧 Your turn 1

Choose a node word central to your own research and run the whole procedure. How many of its top collocates survive reading the concordance lines?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Typically about half. The commonest casualty is a collocate that turns out to
be part of a fixed phrase or a boilerplate formula, which is a fact about the
genre rather than about the concept. Losing half your list to close reading is
not a failure of the method — it is the method working, and it is the specific
thing that distinguishes corpus linguistics from word-counting.

</details>